In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

url = os.getenv("DATABASE_URL")

print(url[:30] if url else "DATABASE_URL NOT FOUND")

postgresql://postgres.mvolsiev


In [10]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    pool_recycle=300
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print(result.fetchone())

(1,)


In [11]:
import pandas as pd

df = pd.read_sql(
    "SELECT * FROM gold.trip_analytics",
    engine
)

print(df.shape)
print(df.columns.tolist())

(174724, 37)
['trip_id', 'full_date', 'day', 'day_name', 'month', 'month_name', 'quarter', 'year', 'week_of_year', 'weekend_flag', 'start_time', 'start_hour', 'start_minute', 'start_second', 'start_time_period', 'end_time', 'end_hour', 'end_minute', 'end_second', 'end_time_period', 'start_station_id', 'start_station_name', 'start_latitude', 'start_longitude', 'end_station_id', 'end_station_name', 'end_latitude', 'end_longitude', 'user_type', 'member_gender', 'member_birth_year', 'member_age', 'age_group', 'bike_id', 'duration_sec', 'duration_min', 'same_station_trip']


In [12]:
route_data = df.dropna(
    subset=["start_station_name", "end_station_name"]
).copy()

route_counts = (
    route_data
    .groupby(["start_station_name", "end_station_name"])
    .size()
    .reset_index(name="trip_count")
)

route_counts["route"] = (
    route_counts["start_station_name"]
    + " → " +
    route_counts["end_station_name"]
)

total_trips = route_counts["trip_count"].sum()

route_counts["pct_of_total"] = (
    route_counts["trip_count"] / total_trips * 100
)

top_routes = (
    route_counts[
        ["route", "trip_count", "pct_of_total"]
    ]
    .sort_values("trip_count", ascending=False)
    .reset_index(drop=True)
)

top_routes.head(12)

,route,trip_count,pct_of_total
0,Berry St at 4th St → San Francisco Ferry Build...,327,0.187152
1,Grand Ave at Perkins St → 19th Street BART Sta...,308,0.176278
2,San Francisco Ferry Building (Harry Bridges Pl...,286,0.163687
3,19th Street BART Station → Grand Ave at Perkin...,283,0.161970
4,The Embarcadero at Sansome St → Steuart St at ...,282,0.161397
5,Townsend St at 7th St → San Francisco Caltrain...,260,0.148806
6,San Fernando St at 7th St → 5th St at Virginia St,249,0.142510
7,Market St at 10th St → Montgomery St BART Stat...,243,0.139076
8,5th St at Virginia St → San Fernando St at 7th St,238,0.136215
9,Market St at 10th St → San Francisco Caltrain ...,232,0.132781


In [13]:
top_destinations = (
    route_counts
    .sort_values(
        ["start_station_name", "trip_count"],
        ascending=[True, False]
    )
    .drop_duplicates("start_station_name")
    .reset_index(drop=True)
)

top_destinations = top_destinations[
    ["start_station_name", "end_station_name", "trip_count"]
]

top_destinations.head(10)

,start_station_name,end_station_name,trip_count
0,10th Ave at E 15th St,Lake Merritt BART Station,9
1,10th St at Fallon St,2nd Ave at E 18th St,110
2,10th St at University Ave,North Berkeley BART Station,25
3,11th St at Bryant St,San Francisco Caltrain Station 2 (Townsend St...,76
4,11th St at Natoma St,San Francisco Caltrain Station 2 (Townsend St...,100
5,13th St at Franklin St,Lakeside Dr at 14th St,22
6,14th St at Filbert St,West Oakland BART Station,58
7,14th St at Mandela Pkwy,West Oakland BART Station,154
8,14th St at Mission St,San Francisco Caltrain Station 2 (Townsend St...,65
9,15th St at Potrero Ave,16th St Mission BART Station 2,55


In [14]:
top_dest_map = dict(
    zip(
        top_destinations["start_station_name"],
        top_destinations["end_station_name"]
    )
)

In [16]:
from pathlib import Path

output_path = Path("Top_Routes.csv")

top_routes.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path.resolve()}")

Saved to: C:\DiskD\AI\Depi-Projects-AI-upload\Data-Analysis\ford_gobike_analysis\Gold_DF\top_routes\Top_Routes.csv
